# CartPole
**Experiments:** DQN and PPO  
**Resume:** automatically loads the latest checkpoint  
**Storage:** only the newest checkpoint is kept

## 1. Setup
Install Gymnasium and Stable-Baselines3.

In [ ]:
%pip install -q "gymnasium[classic-control]>=1.0" "stable-baselines3[extra]>=2.4" "pandas" "matplotlib"

## 2. Configuration
Set the seed, folders, evaluation episodes, and device.

In [ ]:
import re
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym

import torch as th
import torch.nn as nn

from stable_baselines3 import DQN, PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.utils import set_random_seed

SEED = 42
EVAL_EPISODES = 100
TRAIN_CHUNK_STEPS = 25_000

random.seed(SEED)
np.random.seed(SEED)
th.manual_seed(SEED)
set_random_seed(SEED)

DEVICE = "cuda" if th.cuda.is_available() else "cpu"

ROOT = Path("as2_cartpole")
MODEL_DIR = ROOT / "models"
CHECKPOINT_DIR = ROOT / "checkpoints"

for folder in [MODEL_DIR, CHECKPOINT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)

## 3. Training Target
Increase `TOTAL_EPISODES` later if the agent needs more training.  
The notebook resumes from the newest checkpoint.

In [ ]:
TOTAL_EPISODES = 1_000
CHECKPOINT_INTERVAL_EPISODES = 250

## 4. Environment
CartPole has four state variables and two actions. The goal is to balance the pole as long as possible.

In [ ]:
ENV_ID = "CartPole-v1"

def make_env(seed=SEED):
    def _init():
        env = Monitor(
            gym.make(ENV_ID)
        )
        env.reset(seed=seed)
        return env

    return _init

## 5. Custom ANN
The same custom ANN feature extractor is used for DQN and PPO.

In [ ]:
class CustomANN(BaseFeaturesExtractor):
    def __init__(
        self,
        observation_space,
        features_dim=128,
    ):
        super().__init__(
            observation_space,
            features_dim,
        )

        input_dim = int(
            np.prod(observation_space.shape)
        )

        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, features_dim),
            nn.ReLU(),
        )

    def forward(self, observations):
        return self.net(observations)


POLICY_KWARGS = dict(
    features_extractor_class=CustomANN,
    features_extractor_kwargs=dict(
        features_dim=128
    ),
)

## 6. Utilities
Save every 250 episodes and delete the previous checkpoint after the new one is saved.

In [ ]:
def latest_episode_checkpoint(
    folder,
    prefix,
):
    files = list(
        Path(folder).glob(
            f"{prefix}_*_episodes.zip"
        )
    )

    if not files:
        return None, 0

    def get_episode(path):
        match = re.search(
            r"_(\d+)_episodes\.zip$",
            path.name,
        )
        return int(match.group(1)) if match else -1

    latest = max(files, key=get_episode)
    return latest, get_episode(latest)


def load_reward_history(
    checkpoint_path,
):
    reward_path = checkpoint_path.with_name(
        checkpoint_path.stem
        + "_rewards.npy"
    )

    if reward_path.exists():
        return np.load(
            reward_path
        ).tolist()

    return []


class EpisodeCheckpointCallback(BaseCallback):
    def __init__(
        self,
        checkpoint_dir,
        prefix,
        target_episodes,
        start_episode=0,
        existing_rewards=None,
        save_replay_buffer=False,
        verbose=1,
    ):
        super().__init__(verbose)

        self.checkpoint_dir = Path(
            checkpoint_dir
        )
        self.checkpoint_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        self.prefix = prefix
        self.target_episodes = target_episodes
        self.episode_count = start_episode
        self.rewards = list(
            existing_rewards or []
        )
        self.save_replay_buffer = (
            save_replay_buffer
        )

        self.next_checkpoint = (
            (start_episode
             // CHECKPOINT_INTERVAL_EPISODES)
            + 1
        ) * CHECKPOINT_INTERVAL_EPISODES

    def _remove_old_checkpoints(
        self,
        keep_base,
    ):
        for path in self.checkpoint_dir.glob(
            f"{self.prefix}_*_episodes*"
        ):
            if not path.name.startswith(
                keep_base.name
            ):
                path.unlink(missing_ok=True)

    def save_latest(self):
        base = (
            self.checkpoint_dir
            / (
                f"{self.prefix}_"
                f"{self.episode_count}_episodes"
            )
        )

        # Save new files first.
        self.model.save(base)

        np.save(
            str(base) + "_rewards.npy",
            np.asarray(
                self.rewards,
                dtype=np.float32,
            ),
        )

        if (
            self.save_replay_buffer
            and hasattr(
                self.model,
                "save_replay_buffer",
            )
        ):
            self.model.save_replay_buffer(
                str(base)
                + "_replay_buffer.pkl"
            )

        # Delete older checkpoint files.
        self._remove_old_checkpoints(base)

        if self.verbose:
            print(
                f"Latest checkpoint saved at "
                f"episode {self.episode_count}"
            )

    def _on_step(self):
        for info in self.locals.get(
            "infos",
            [],
        ):
            ep_info = info.get("episode")

            if ep_info is None:
                continue

            self.episode_count += 1
            self.rewards.append(
                float(ep_info["r"])
            )

            if (
                self.episode_count
                >= self.next_checkpoint
            ):
                self.save_latest()

                while (
                    self.next_checkpoint
                    <= self.episode_count
                ):
                    self.next_checkpoint += (
                        CHECKPOINT_INTERVAL_EPISODES
                    )

            if (
                self.episode_count
                >= self.target_episodes
            ):
                return False

        return True


def train_until_target(
    model,
    callback,
):
    while (
        callback.episode_count
        < TOTAL_EPISODES
    ):
        model.learn(
            total_timesteps=TRAIN_CHUNK_STEPS,
            callback=callback,
            reset_num_timesteps=False,
            progress_bar=True,
        )

        print(
            f"Completed episodes: "
            f"{callback.episode_count}/"
            f"{TOTAL_EPISODES}"
        )

    # Save the exact latest state and remove older checkpoint.
    callback.save_latest()


def evaluate_model(model, env):
    rewards, lengths = evaluate_policy(
        model,
        env,
        n_eval_episodes=EVAL_EPISODES,
        deterministic=True,
        return_episode_rewards=True,
        warn=False,
    )

    return {
        "mean_reward": float(np.mean(rewards)),
        "std_reward": float(np.std(rewards)),
        "median_reward": float(np.median(rewards)),
        "mean_length": float(np.mean(lengths)),
    }


def plot_rewards(rewards, title):
    series = pd.Series(
        rewards,
        dtype=float,
    )

    plt.figure(figsize=(9, 4))

    plt.plot(
        series,
        alpha=0.35,
        label="Episode reward",
    )

    if len(series) >= 100:
        plt.plot(
            series.rolling(100).mean(),
            label="100-episode mean",
        )

    plt.title(title)
    plt.xlabel("Episode")
    plt.ylabel("Reward")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()

## 7. Random Baseline
Evaluate a random policy before training.

In [ ]:
def random_baseline():
    returns = []

    for ep in range(EVAL_EPISODES):
        env = gym.make(ENV_ID)
        obs, _ = env.reset(
            seed=SEED + ep
        )

        total = 0.0
        terminated = False
        truncated = False

        while not (
            terminated or truncated
        ):
            action = env.action_space.sample()

            (
                obs,
                reward,
                terminated,
                truncated,
                _,
            ) = env.step(action)

            total += float(reward)

        env.close()
        returns.append(total)

    return {
        "mean_reward": float(np.mean(returns)),
        "std_reward": float(np.std(returns)),
        "median_reward": float(np.median(returns)),
    }


random_result = random_baseline()
random_result

## 8. Experiment 3 — DQN
DQN automatically resumes from the latest saved episode checkpoint.

In [ ]:
dqn_env = DummyVecEnv([
    make_env(SEED)
])

dqn_dir = CHECKPOINT_DIR / "dqn"

dqn_ckpt, completed = (
    latest_episode_checkpoint(
        dqn_dir,
        "cartpole_dqn",
    )
)

if dqn_ckpt is None:
    print(
        "No DQN checkpoint found. "
        "Starting from 0."
    )

    dqn = DQN(
        "MlpPolicy",
        dqn_env,
        learning_rate=1e-3,
        buffer_size=50_000,
        learning_starts=1_000,
        batch_size=64,
        gamma=0.99,
        train_freq=4,
        gradient_steps=1,
        target_update_interval=500,
        exploration_fraction=0.20,
        exploration_initial_eps=1.0,
        exploration_final_eps=0.02,
        policy_kwargs=POLICY_KWARGS,
        verbose=1,
        seed=SEED,
        device=DEVICE,
    )

    old_rewards = []

else:
    print("Resuming DQN from:", dqn_ckpt)
    print("Completed episodes:", completed)

    dqn = DQN.load(
        dqn_ckpt,
        env=dqn_env,
        device=DEVICE,
    )

    replay_path = dqn_ckpt.with_name(
        dqn_ckpt.stem
        + "_replay_buffer.pkl"
    )

    if replay_path.exists():
        dqn.load_replay_buffer(
            replay_path
        )
        print("Replay buffer loaded.")

    old_rewards = load_reward_history(
        dqn_ckpt
    )

dqn_callback = EpisodeCheckpointCallback(
    checkpoint_dir=dqn_dir,
    prefix="cartpole_dqn",
    target_episodes=TOTAL_EPISODES,
    start_episode=completed,
    existing_rewards=old_rewards,
    save_replay_buffer=True,
)

if completed < TOTAL_EPISODES:
    train_until_target(
        dqn,
        dqn_callback,
    )
else:
    print(
        "DQN already reached the target. "
        "Increase TOTAL_EPISODES to train more."
    )

dqn.save(
    MODEL_DIR / "cartpole_dqn_final"
)

### DQN Evaluation
Evaluate the current DQN model on separate episodes.

In [ ]:
plot_rewards(
    dqn_callback.rewards,
    "CartPole DQN",
)

dqn_eval_env = DummyVecEnv([
    make_env(SEED + 1000)
])

dqn_result = evaluate_model(
    dqn,
    dqn_eval_env,
)

dqn_result

## 9. Experiment 4 — PPO
PPO resumes in the same way and keeps only its newest checkpoint.

In [ ]:
ppo_env = DummyVecEnv([
    make_env(SEED + 1)
])

ppo_dir = CHECKPOINT_DIR / "ppo"

ppo_ckpt, completed = (
    latest_episode_checkpoint(
        ppo_dir,
        "cartpole_ppo",
    )
)

if ppo_ckpt is None:
    print(
        "No PPO checkpoint found. "
        "Starting from 0."
    )

    ppo = PPO(
        "MlpPolicy",
        ppo_env,
        learning_rate=3e-4,
        n_steps=1024,
        batch_size=64,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        policy_kwargs=POLICY_KWARGS,
        verbose=1,
        seed=SEED,
        device=DEVICE,
    )

    old_rewards = []

else:
    print("Resuming PPO from:", ppo_ckpt)
    print("Completed episodes:", completed)

    ppo = PPO.load(
        ppo_ckpt,
        env=ppo_env,
        device=DEVICE,
    )

    old_rewards = load_reward_history(
        ppo_ckpt
    )

ppo_callback = EpisodeCheckpointCallback(
    checkpoint_dir=ppo_dir,
    prefix="cartpole_ppo",
    target_episodes=TOTAL_EPISODES,
    start_episode=completed,
    existing_rewards=old_rewards,
    save_replay_buffer=False,
)

if completed < TOTAL_EPISODES:
    train_until_target(
        ppo,
        ppo_callback,
    )
else:
    print(
        "PPO already reached the target. "
        "Increase TOTAL_EPISODES to train more."
    )

ppo.save(
    MODEL_DIR / "cartpole_ppo_final"
)

### PPO Evaluation
Evaluate the current PPO model on separate episodes.

In [ ]:
plot_rewards(
    ppo_callback.rewards,
    "CartPole PPO",
)

ppo_eval_env = DummyVecEnv([
    make_env(SEED + 1001)
])

ppo_result = evaluate_model(
    ppo,
    ppo_eval_env,
)

ppo_result

## 10. Results
Compare Random, DQN, and PPO.

In [ ]:
cartpole_results = pd.DataFrame([
    {
        "Algorithm": "Random",
        **random_result,
    },
    {
        "Algorithm": "DQN",
        **dqn_result,
    },
    {
        "Algorithm": "PPO",
        **ppo_result,
    },
])

cartpole_results